### Chatbot with Conversational History & Session Management using LCEL

In [ ]:
from operator import itemgetter

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.chat_message_histories import ChatMessageHistory

from langchain_core.documents import Document
from langchain_core.messages import trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

#### STEP 1: INITIALIZATION

In [ ]:
load_dotenv()

model = ChatGroq(model="llama-3.1-8b-instant")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#### STEP 2: VECTOR STORE & RETRIEVER

In [ ]:
raw_documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal_doc"},
    ),
    Document(
        page_content="Cats are independent creatures that often enjoy their own space.",
        metadata={"source": "mammal_doc"},
    ),
    Document(
        page_content="Goldfish are popular low-maintenance pets for small aquariums.",
        metadata={"source": "aquatic_doc"},
    ),
    Document(
        page_content="Sharks are apex predators playing a vital role in marine ecosystems.",
        metadata={"source": "aquatic_doc"},
    ),
]

vector_store = Chroma.from_documents(
    documents=raw_documents, embedding=embeddings, collection_name="pet_knowledge"
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # Increased for better context
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#### STEP 3: HISTORY TRIMMER

In [ ]:
history_trimmer = trim_messages(
    max_tokens=500,  # Increased for better conversation memory
    strategy="last",
    token_counter=model,
    include_system=False,
    allow_partial=False,
    start_on="human",
)

#### STEP 4: PROMPT

In [ ]:
prompt_recipe = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful and intelligent AI assistant.
Answer the user's question using the provided context and conversation history.
If the answer is not available in the context or history, honestly say that you don't know.

You must respond entirely in this language: {language}.

Context:
{context}""",
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

#### STEP 5: LCEL CHAIN

In [ ]:
base_rag_chain = (
    RunnablePassthrough.assign(
        chat_history=itemgetter("chat_history") | history_trimmer,
        context=itemgetter("question") | retriever | format_docs,
        # language is passed through automatically, but we make it explicit for clarity
        language=itemgetter("language"),
    )
    | prompt_recipe
    | model
)

#### STEP 6: STATEFUL MULTI-SESSION MANAGEMENT

In [ ]:
session_database = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_database:
        session_database[session_id] = ChatMessageHistory()
    return session_database[session_id]


chatbot = RunnableWithMessageHistory(
    base_rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

#### STEP 7: TESTING

In [ ]:
print("=== RAG Chatbot with Memory & Multilingual Support ===\n")

# Session 1
config1 = {"configurable": {"session_id": "user_session_alpha"}}

print("--- Session 1: Turn 1 ---")
response1 = chatbot.invoke(
    {
        "question": "Hi, my name is Krish. Tell me about dogs.", "language": "English"
    },
    config=config1,
)
print(f"Bot: {response1.content}\n")

In [ ]:
print("--- Session 1: Turn 2 ---")
response2 = chatbot.invoke(
    {
        "question": "What is my name and what animal did we talk about?",
        "language": "English",
    },
    config=config1,
)
print(f"Bot: {response2.content}\n")

In [ ]:
# Session 2 (Isolated)
config2 = {"configurable": {"session_id": "user_session_beta"}}

print("--- Session 2: New Session Test ---")
response3 = chatbot.invoke(
        {
            "question": "What is my name?", "language": "English"
        }, 
        config=config2
)
print(f"Bot: {response3.content}\n")